# v2: Evidence-Aware Deep Research Agent

v2 在 v1 的多轮工具调用基础上继续改进：将局部文档定位工具下沉到 `agent/tools.py`，新增候选证据片段收集工具，并在 agent loop 中显式维护 evidence table、query-result signature 和最终验证阶段。

本 notebook 仍然只用于服务器环境运行。本地不要执行、不要安装依赖、不要构建索引、不要启动或调用 vLLM。

## 1. 配置与导入

v2 使用 `get_research_tool_specs_and_registry()` 获取扩展后的工具集合，包括 `search`、`get_document`、`get_document_window`、`find_in_document` 和 `collect_evidence_snippets`。

In [ ]:
from pathlib import Path
import json
import re
import sys
from typing import Any, Dict, List, Optional

project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from agent.dataset_utils import load_jsonl
from agent.eval import run_evaluation
from agent.tools import build_searcher, get_research_tool_specs_and_registry
from agent.vllm_client import VLLMClient

VLLM_BASE_URL = "http://127.0.0.1:8000/v1"
MODEL_NAME = "qwen_auto"
API_KEY = "dummy"

BM25_INDEX_PATH = str(project_root / "indexes" / "browsecomp_plus_bm25.sqlite")
HARD50_PATH = str(project_root / "browsecomp_plus_hard50.jsonl")
SUBMISSION_PATH = str(project_root / "runs" / "v2_submission.jsonl")
EVAL_OUTPUT_PATH = str(project_root / "runs" / "v2_eval_results.jsonl")

client = VLLMClient(base_url=VLLM_BASE_URL, api_key=API_KEY)
searcher = build_searcher(index_path=BM25_INDEX_PATH)
tool_specs, tool_registry = get_research_tool_specs_and_registry(
    searcher=searcher,
    k=8,
    snippet_max_chars=1400,
    document_window_chars=2200,
)


## 2. Prompt 设计

v2 的 prompt 比 v1 更强调“证据驱动”：先构造线索、再搜索、再打开/定位文档，最终必须基于 evidence table 输出短答案。

In [ ]:
SYSTEM_PROMPT = """
You are an evidence-aware Deep Research Agent for BrowseComp-Plus.
You must answer using only evidence returned by the local BrowseComp-Plus tools.

Tools:
- search(query): retrieve BM25 snippets from the fixed corpus.
- get_document(docid): open a full document when a top result looks central.
- get_document_window(docid, start, window_chars): read a bounded span from a long document.
- find_in_document(docid, keyword): locate a clue, entity, date, title, or phrase inside a document.
- collect_evidence_snippets(docids, keywords): collect snippets for final evidence checking.

Research policy:
1. Convert the question into searchable clues and subgoals.
2. Search with focused queries. Avoid repeating queries that returned the same docids.
3. Open or inspect promising documents before treating a clue as confirmed.
4. Maintain candidate answers, supporting docids, and unresolved gaps.
5. Before finalizing, verify the candidate answer against at least one supporting snippet when possible.
6. Do not use external knowledge. Do not guess from memory.

Final answer format:
Explanation: <brief reasoning grounded in evidence>
Evidence: <docid: short support summary; repeat if needed>
Exact Answer: <short final answer only>
Confidence: <0-100>%
""".strip()


FINALIZER_PROMPT = """
Finalize now. Do not call more tools.
Use only the compressed state and tool evidence already present in the conversation.
If evidence is incomplete, state the limitation in Explanation and lower Confidence.
Return exactly:
Explanation: ...
Evidence: ...
Exact Answer: ...
Confidence: ...%
""".strip()


## 3. 状态、去重与工具执行辅助函数

v2 不只检查 query 是否重复，还记录每次 search 返回的 docid 签名。即使模型改写了 query，只要返回集合基本相同，也会被视为低增益搜索。

In [ ]:
def canonical_text(text: str) -> str:
    return re.sub(r"\s+", " ", str(text or "").strip().lower())


def truncate_text(text: str, max_chars: int) -> str:
    text = str(text or "")
    if len(text) <= max_chars:
        return text
    return text[:max_chars].rstrip() + "... [truncated]"


def stable_json(data: Any, max_chars: Optional[int] = None) -> str:
    text = json.dumps(data, ensure_ascii=False, sort_keys=True)
    return truncate_text(text, max_chars) if max_chars else text


def parse_tool_arguments(raw_arguments: Any) -> Dict[str, Any]:
    if isinstance(raw_arguments, dict):
        return raw_arguments
    if raw_arguments in (None, ""):
        return {}
    try:
        parsed = json.loads(raw_arguments)
    except json.JSONDecodeError:
        return {"_argument_parse_error": str(raw_arguments)}
    return parsed if isinstance(parsed, dict) else {"_argument_parse_error": str(raw_arguments)}


def normalize_tool_result(tool_name: str, result: Any, max_document_chars: int = 4500) -> Any:
    if tool_name == "get_document" and isinstance(result, dict):
        normalized = dict(result)
        if "text" in normalized:
            full_text = str(normalized.get("text", ""))
            normalized["text"] = truncate_text(full_text, max_document_chars)
            normalized["text_length"] = len(full_text)
            normalized["text_is_truncated"] = len(full_text) > max_document_chars
        return normalized
    return result


def execute_tool_call(tool_call: Dict[str, Any]) -> Dict[str, Any]:
    function = tool_call.get("function", {}) or {}
    name = function.get("name", "")
    arguments = parse_tool_arguments(function.get("arguments", "{}"))
    if "_argument_parse_error" in arguments:
        return {"ok": False, "tool_name": name, "arguments": arguments, "error": "invalid JSON arguments"}
    if name not in tool_registry:
        return {"ok": False, "tool_name": name, "arguments": arguments, "error": "unknown tool"}
    try:
        raw_result = tool_registry[name](**arguments)
        result = normalize_tool_result(name, raw_result)
        return {"ok": True, "tool_name": name, "arguments": arguments, "tool_result": result}
    except Exception as exc:
        return {"ok": False, "tool_name": name, "arguments": arguments, "error": repr(exc)}


def make_tool_message(tool_call: Dict[str, Any], executed: Dict[str, Any]) -> Dict[str, Any]:
    content = executed.get("tool_result") if executed.get("ok") else executed
    return {
        "role": "tool",
        "tool_call_id": tool_call.get("id", ""),
        "content": json.dumps(content, ensure_ascii=False),
    }


def docids_from_tool_result(tool_name: str, result: Any) -> List[str]:
    if tool_name == "search" and isinstance(result, list):
        return [str(item.get("docid", "")) for item in result if isinstance(item, dict) and item.get("docid")]
    if isinstance(result, dict) and result.get("docid"):
        return [str(result.get("docid"))]
    if tool_name == "collect_evidence_snippets" and isinstance(result, dict):
        return [str(item.get("docid", "")) for item in result.get("snippets", []) if item.get("docid")]
    return []


def search_signature(result: Any) -> str:
    docids = docids_from_tool_result("search", result)
    return ",".join(docids[:8])


def initial_state(question: str) -> Dict[str, Any]:
    return {
        "question": question,
        "searched_queries": [],
        "searched_query_keys": [],
        "search_signatures": [],
        "seen_docids": [],
        "opened_docids": [],
        "evidence_table": [],
        "tool_events": [],
        "rounds": [],
        "low_gain_rounds": 0,
    }


def summarize_tool_result(tool_name: str, result: Any) -> str:
    if tool_name == "search":
        return "search docids=" + ", ".join(docids_from_tool_result(tool_name, result)[:8])
    if tool_name == "collect_evidence_snippets" and isinstance(result, dict):
        return f"collected {len(result.get('snippets', []))} evidence snippets"
    if tool_name == "find_in_document" and isinstance(result, dict):
        return f"{result.get('num_matches', 0)} matches for {result.get('keyword', '')} in {result.get('docid', '')}"
    if isinstance(result, dict) and result.get("docid"):
        return f"inspected docid={result.get('docid', '')}"
    return truncate_text(stable_json(result), 300)


def update_state_from_execution(state: Dict[str, Any], executed: Dict[str, Any], round_id: int) -> int:
    tool_name = executed.get("tool_name", "")
    arguments = executed.get("arguments", {}) or {}
    result = executed.get("tool_result")
    new_signal = 0

    if tool_name == "search":
        query = str(arguments.get("query", ""))
        query_key = canonical_text(query)
        if query_key and query_key not in state["searched_query_keys"]:
            state["searched_query_keys"].append(query_key)
            state["searched_queries"].append(query)
            new_signal += 1
        signature = search_signature(result)
        if signature and signature not in state["search_signatures"]:
            state["search_signatures"].append(signature)
            new_signal += 1

    for docid in docids_from_tool_result(tool_name, result):
        if docid and docid not in state["seen_docids"]:
            state["seen_docids"].append(docid)
            new_signal += 1
        if tool_name in {"get_document", "get_document_window", "find_in_document"} and docid not in state["opened_docids"]:
            state["opened_docids"].append(docid)
            new_signal += 1

    if tool_name in {"find_in_document", "collect_evidence_snippets"}:
        state["evidence_table"].append(
            {
                "round_id": round_id,
                "tool_name": tool_name,
                "arguments": arguments,
                "summary": summarize_tool_result(tool_name, result),
            }
        )

    state["tool_events"].append(
        {
            "round_id": round_id,
            "tool_name": tool_name,
            "arguments": arguments,
            "ok": executed.get("ok", False),
            "summary": summarize_tool_result(tool_name, result),
        }
    )
    return new_signal


def public_state_summary(state: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "searched_queries": state["searched_queries"][-12:],
        "recent_search_signatures": state["search_signatures"][-8:],
        "seen_docids": state["seen_docids"][-30:],
        "opened_docids": state["opened_docids"][-20:],
        "evidence_table": state["evidence_table"][-12:],
        "recent_tool_events": state["tool_events"][-12:],
        "low_gain_rounds": state["low_gain_rounds"],
    }


def extract_exact_answer(text: str) -> str:
    text = str(text or "").strip()
    match = re.search(r"(?im)^\s*Exact Answer\s*:\s*(.+?)\s*$", text)
    if match:
        return match.group(1).strip()
    match = re.search(r"(?im)^\s*Answer\s*:\s*(.+?)\s*$", text)
    return match.group(1).strip() if match else text


## 4. v2 Agent Loop

v2 仍保留完整 `messages` 轨迹，但每轮实际送入模型的是：原始问题、压缩状态、最近若干条消息。达到最大轮数、低增益轮数过多，或模型停止调用工具时，会进入 finalizer。

In [ ]:
def build_model_messages(
    messages: List[Dict[str, Any]],
    state: Dict[str, Any],
    recent_message_limit: int = 14,
) -> List[Dict[str, Any]]:
    state_message = {
        "role": "user",
        "content": (
            "Compressed research state. Use it to plan the next useful tool call and avoid low-gain repetition.\n"
            + json.dumps(public_state_summary(state), ensure_ascii=False, indent=2)
        ),
    }
    recent = messages[2:]
    if len(recent) > recent_message_limit:
        recent = recent[-recent_message_limit:]
        while recent and recent[0].get("role") == "tool":
            recent = recent[1:]
    return [messages[0], messages[1], state_message] + recent


def response_to_assistant_message(response: Dict[str, Any]) -> tuple[Dict[str, Any], List[Dict[str, Any]]]:
    message = response["choices"][0]["message"]
    assistant_message = {"role": "assistant", "content": message.get("content") or ""}
    tool_calls = message.get("tool_calls") or []
    if tool_calls:
        assistant_message["tool_calls"] = tool_calls
    return assistant_message, tool_calls


def finalize_answer(
    messages: List[Dict[str, Any]],
    state: Dict[str, Any],
    reason: str,
    max_tokens: int = 900,
) -> str:
    final_instruction = {
        "role": "user",
        "content": reason + "\n\n" + FINALIZER_PROMPT,
    }
    response = client.simple_chat(
        model=MODEL_NAME,
        messages=build_model_messages(messages, state) + [final_instruction],
        temperature=0.0,
        max_tokens=max_tokens,
    )
    assistant_message, _ = response_to_assistant_message(response)
    messages.append(final_instruction)
    messages.append({"role": "assistant", "content": assistant_message.get("content", "")})
    return assistant_message.get("content", "")


def run_v2_agent(
    question: str,
    query_id: Optional[str] = None,
    max_rounds: int = 8,
    low_gain_round_limit: int = 2,
    max_tokens: int = 1300,
    recent_message_limit: int = 14,
) -> Dict[str, Any]:
    messages: List[Dict[str, Any]] = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    state = initial_state(question)
    final_output = ""
    status = "max_rounds_reached"

    for round_id in range(1, max_rounds + 1):
        response = client.simple_chat(
            model=MODEL_NAME,
            messages=build_model_messages(messages, state, recent_message_limit=recent_message_limit),
            temperature=0.0,
            max_tokens=max_tokens,
            tools=tool_specs,
            tool_choice="auto",
        )
        assistant_message, tool_calls = response_to_assistant_message(response)
        messages.append(assistant_message)
        state["rounds"].append(
            {
                "round_id": round_id,
                "assistant_content": truncate_text(assistant_message.get("content", ""), 500),
                "tool_call_count": len(tool_calls),
            }
        )

        if not tool_calls:
            status = "completed_without_more_tools"
            final_output = finalize_answer(
                messages,
                state,
                reason="The model stopped calling tools. Produce the final answer with current evidence.",
            )
            break

        round_signal = 0
        for tool_call in tool_calls:
            executed = execute_tool_call(tool_call)
            messages.append(make_tool_message(tool_call, executed))
            round_signal += update_state_from_execution(state, executed, round_id)

        if round_signal == 0:
            state["low_gain_rounds"] += 1
        else:
            state["low_gain_rounds"] = 0

        if state["low_gain_rounds"] >= low_gain_round_limit:
            status = "stopped_low_gain_search"
            final_output = finalize_answer(
                messages,
                state,
                reason="Recent tool calls did not add new queries, docids, or evidence. Stop searching.",
            )
            break
    else:
        final_output = finalize_answer(
            messages,
            state,
            reason=f"Reached max_rounds={max_rounds}. Stop searching and answer from gathered evidence.",
        )

    return {
        "query_id": query_id,
        "status": status,
        "predicted_answer": extract_exact_answer(final_output),
        "final_output": final_output,
        "messages": messages,
        "state_summary": public_state_summary(state),
    }


## 5. 生成 submission 与评测辅助函数

这些函数仅供服务器环境执行。`messages` 保留完整轨迹，`state_summary` 作为额外调试信息保留在每条记录里。

In [ ]:
def build_submission_record(row: Dict[str, Any], **agent_kwargs: Any) -> Dict[str, Any]:
    result = run_v2_agent(
        question=row["query"],
        query_id=str(row.get("query_id", "")),
        **agent_kwargs,
    )
    return {
        "query_id": str(row.get("query_id", "")),
        "predicted_answer": result["predicted_answer"],
        "status": result["status"],
        "messages": result["messages"],
        "state_summary": result["state_summary"],
    }


def generate_submission(
    dataset_path: str = HARD50_PATH,
    output_path: str = SUBMISSION_PATH,
    limit: Optional[int] = 50,
    **agent_kwargs: Any,
) -> List[Dict[str, Any]]:
    rows = load_jsonl(dataset_path, limit=limit)
    output_file = Path(output_path)
    output_file.parent.mkdir(parents=True, exist_ok=True)

    records = []
    with output_file.open("w", encoding="utf-8") as fout:
        for index, row in enumerate(rows, start=1):
            print(f"[{index}/{len(rows)}] query_id={row.get('query_id', '')}")
            record = build_submission_record(row, **agent_kwargs)
            records.append(record)
            fout.write(json.dumps(record, ensure_ascii=False) + "\n")

    print(f"Saved {len(records)} records to {output_path}")
    return records


def evaluate_submission(
    submission_path: str = SUBMISSION_PATH,
    dataset_path: str = HARD50_PATH,
    output_path: str = EVAL_OUTPUT_PATH,
):
    return run_evaluation(
        submission_path=submission_path,
        dataset_path=dataset_path,
        model_name=MODEL_NAME,
        base_url=VLLM_BASE_URL,
        api_key=API_KEY,
        output_path=output_path,
        verbose=True,
    )


## 6. 服务器运行入口

本地不要执行。到服务器确认 vLLM 和 BM25 索引已就绪后，再取消注释运行。

```python
# records = generate_submission(limit=50, max_rounds=8, low_gain_round_limit=2)
# summary, details = evaluate_submission()
```